# Extrapolation Benchmark: THP vs RoTHP vs HoTHP

Treina e avalia os modelos THP, RoTHP e HoTHP em datasets reais com **extrapolação de comprimento**.

Os modelos são treinados com sequências de comprimento `L_train` e avaliados em comprimentos
`L_test = ratio * L_train`, com `ratio ∈ {1, 2, 5, 10}`.  
A métrica de avaliação é o **NLL médio por evento** (Negative Log-Likelihood, menor = melhor).

## Datasets

| Dataset | Pasta | L_train | Fonte |
|---------|-------|---------|-------|
| stackoverflow | `so` | 10 | Drive público (Célula 3) |
| retweet | `retweet` | 10 | Drive público (Célula 3) |
| memetrack | `meme` | 50 | Drive público (Célula 3) |
| mimic-ii | `mimic` | 10 | Drive público (Célula 3) |

Não é necessário editar nenhuma variável — execute as células na ordem.

## Célula 1 — Instalação

In [ ]:
import os, sys

# ── Clona o repositório ufc-easytpp ──────────────────────────────
if not os.path.exists('ufc-easytpp'):
    !git clone https://github.com/hugoramos/ufc-easytpp.git

sys.path.insert(0, 'ufc-easytpp')

# ── Instala dependências ──────────────────────────────────────────
!pip install omegaconf datasets pyyaml matplotlib pandas seaborn tqdm gdown -q

# ── Fix 1: __init__.py mínimo (evita imports de modelos não usados) ──
_init_path = 'ufc-easytpp/easy_tpp/model/__init__.py'
with open(_init_path, 'w') as f:
    f.write(
        "from easy_tpp.model.torch_model.torch_basemodel import TorchBaseModel\n"
        "from easy_tpp.model.torch_model.torch_thp    import THP    as TorchTHP\n"
        "from easy_tpp.model.torch_model.torch_rothp  import RoTHP  as TorchRoTHP\n"
        "from easy_tpp.model.torch_model.torch_hothp  import HoTHP  as TorchHoTHP\n"
    )

# ── Fix 2: remove import de notebook local no torch_hothp.py ─────
_hothp_path = 'ufc-easytpp/easy_tpp/model/torch_model/torch_hothp.py'
with open(_hothp_path, 'r') as f:
    code = f.read()
if 'from notebooks.' in code:
    code = code.replace(
        'from notebooks.Extrapolation_and_Attention_Analysis import attention_fixed\n', '')
    with open(_hothp_path, 'w') as f:
        f.write(code)

import torch
GPU = 0 if torch.cuda.is_available() else -1
print(f'OK — GPU={GPU}')


## Célula 2 — Configuração do Experimento

Ajuste os parâmetros globais e os caminhos dos datasets manuais aqui.

In [ ]:
import os, torch

# ----------------------------------------------------------------
# Parâmetros globais
# ----------------------------------------------------------------
L_TRAIN              = 10              # comprimento de treino (eventos por sequência)
EXTRAPOLATION_RATIOS = [1, 2, 5, 10]  # L_test = ratio * L_train

MODELS  = ['THP', 'RoTHP', 'HoTHP']
BASE_DIR    = './checkpoints'
RESULTS_DIR = './results'
DATA_DIR    = './data'
NHP_RAW_DIR = './nhp_raw'
for d in [BASE_DIR, RESULTS_DIR, DATA_DIR, NHP_RAW_DIR]:
    os.makedirs(d, exist_ok=True)

GPU = 0 if torch.cuda.is_available() else -1

# ----------------------------------------------------------------
# Hiperparâmetros de treino (iguais para todos os modelos)
# ----------------------------------------------------------------
HIDDEN_SIZE   = 64
TIME_EMB_SIZE = 16
NUM_LAYERS    = 2
NUM_HEADS     = 4
DROPOUT       = 0.1
BATCH_SIZE    = 256
MAX_EPOCHS    = 30
LEARNING_RATE = 1e-3

# ----------------------------------------------------------------
# Dicionário DATASETS
#
# 'nhp_subdir' → nome da subpasta em NHP_RAW_DIR (conforme Drive público)
# 'nhp_fold'   → subfold a usar (None = sem folds)
# 'pkl_dir'    → override manual (sobrepõe nhp_subdir se preenchido)
# 'num_types'  → número de tipos de evento (None = detectado automaticamente)
# 'l_train'    → override de L_train por dataset (None = usa L_TRAIN global)
# ----------------------------------------------------------------
DATASETS = {
    'stackoverflow': {
        'nhp_subdir': 'so',
        'nhp_fold':   None,
        'pkl_dir':    None,
        'num_types':  None,
        'l_train':    None,
        'note':       'NeuralHawkes (so). max_len≈736.',
    },
    'retweet': {
        'nhp_subdir': 'retweet',
        'nhp_fold':   None,
        'pkl_dir':    None,
        'num_types':  None,
        'l_train':    None,
        'note':       'NeuralHawkes (retweet).',
    },
    'memetrack': {
        'nhp_subdir': 'meme',
        'nhp_fold':   None,
        'pkl_dir':    None,
        'num_types':  None,
        'l_train':    50,
        'note':       'NeuralHawkes (meme).',
    },
    'mimic-ii': {
        'nhp_subdir': 'mimic',
        'nhp_fold':   None,
        'pkl_dir':    None,
        'num_types':  None,
        'l_train':    None,
        'note':       'NeuralHawkes (mimic). Sequências curtas.',
    },
}

print("Configuração do experimento:")
print(f"  L_train global  : {L_TRAIN}")
print(f"  Ratios (L_test) : {[(r, r*L_TRAIN) for r in EXTRAPOLATION_RATIOS]}")
print(f"  Modelos         : {MODELS}")
print(f"  GPU             : {GPU}")
print()
for ds_name, cfg in DATASETS.items():
    lt = cfg['l_train'] or L_TRAIN
    print(f"  {ds_name:15s} l_train={lt:3d}  L_test_max={lt*max(EXTRAPOLATION_RATIOS):4d}  {cfg['note']}")

## Célula 3 — Download dos Datasets

Os datasets são baixados automaticamente do Drive público via `gdown`.  
Nenhuma ação manual é necessária — basta executar a célula.

In [ ]:
import os, subprocess, shutil

DRIVE_FOLDER_ID  = '1bxwawkeRmsPpDmJH0oMeYREno5NZttxu'
DRIVE_FOLDER_URL = f'https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}'

os.makedirs(NHP_RAW_DIR, exist_ok=True)

def _has_pkl(path):
    """Retorna True se o diretório contiver ao menos um .pkl em qualquer nível."""
    for _, _, files in os.walk(path):
        if any(f.endswith('.pkl') for f in files):
            return True
    return False

already = [d for d in os.listdir(NHP_RAW_DIR)
           if os.path.isdir(os.path.join(NHP_RAW_DIR, d))
           and _has_pkl(os.path.join(NHP_RAW_DIR, d))]

if already:
    print(f"Datasets já presentes em {NHP_RAW_DIR}: {already}")
else:
    print(f"Baixando pasta pública do Drive...\n  {DRIVE_FOLDER_URL}\n")
    r = subprocess.run(
        ['gdown', '--folder', DRIVE_FOLDER_URL, '-O', NHP_RAW_DIR, '--remaining-ok'],
        capture_output=True, text=True
    )
    if r.stdout:
        print(r.stdout[-3000:])
    if r.returncode != 0:
        print("STDERR:", r.stderr[-500:])
        raise RuntimeError(
            "Download falhou.\n"
            "Confirme que a pasta é pública (Anyone with the link → Viewer).\n"
            f"URL: {DRIVE_FOLDER_URL}"
        )

    # Algumas versões do gdown criam uma subpasta wrapper dentro do destino.
    # Se NHP_RAW_DIR contém apenas 1 subdir e ele não tem pkl direto, "achata".
    subdirs = [d for d in os.listdir(NHP_RAW_DIR)
               if os.path.isdir(os.path.join(NHP_RAW_DIR, d))]
    if len(subdirs) == 1:
        wrapper = os.path.join(NHP_RAW_DIR, subdirs[0])
        inner   = os.listdir(wrapper)
        if not any(f.endswith('.pkl') for f in inner):   # wrapper sem pkl → move conteúdo
            for item in inner:
                shutil.move(os.path.join(wrapper, item),
                            os.path.join(NHP_RAW_DIR, item))
            os.rmdir(wrapper)
            print(f"  Movido conteúdo de '{subdirs[0]}/' para {NHP_RAW_DIR}/")

print(f"\nConteúdo de {NHP_RAW_DIR}:")
for item in sorted(os.listdir(NHP_RAW_DIR)):
    p = os.path.join(NHP_RAW_DIR, item)
    if os.path.isdir(p):
        files = sorted(os.listdir(p))
        print(f"  {item}/  →  {files[:8]}")

In [ ]:
import pickle, shutil, os
import numpy as np
from datasets import load_dataset


# ----------------------------------------------------------------
# Detecta e normaliza o formato do evento NHP
#
# Os pkls do NeuralHawkes Drive guardam eventos como dicts com as chaves:
#   'time_since_start', 'time_since_last_event', 'type_event'
# — exatamente o formato que o EasyTPP espera.
#
# Mas, para robustez, tratamos também o caso em que eventos são listas
# [time_since_start, time_since_last_event, type_event] ou pares [dt, type].
# ----------------------------------------------------------------
def normalize_event(event, t_prev=0.0):
    """Converte qualquer formato de evento para dict EasyTPP."""
    if isinstance(event, dict):
        return {
            'time_since_start':      float(event.get('time_since_start',
                                           event.get('time_since_last_event', 0))),
            'time_since_last_event': float(event.get('time_since_last_event', 0)),
            'type_event':            int(event.get('type_event',
                                        event.get('event_type', 0))),
        }
    # lista ou tupla
    event = list(event)
    if len(event) == 3:
        return {
            'time_since_start':      float(event[0]),
            'time_since_last_event': float(event[1]),
            'type_event':            int(event[2]),
        }
    else:   # [dt, type]
        t = t_prev + float(event[0])
        return {
            'time_since_start':      t,
            'time_since_last_event': float(event[0]),
            'type_event':            int(event[1]),
        }


def normalize_sequences(raw_seqs):
    """Normaliza uma lista de sequências brutas para formato EasyTPP."""
    result = []
    for seq in raw_seqs:
        norm, t = [], 0.0
        for ev in seq:
            e = normalize_event(ev, t)
            t = e['time_since_start']
            norm.append(e)
        result.append(norm)
    return result


def nhp_pkl_to_easytpp(src_pkl: str, split: str, dst_pkl: str) -> int:
    """
    Carrega um pkl do NeuralHawkes Drive (que já tem split = 'train'/'dev'/'test')
    e salva no formato EasyTPP: {'dim_process': int, split: [seqs]}.

    Retorna dim_process.
    """
    with open(src_pkl, 'rb') as f:
        raw = pickle.load(f)

    # O pkl NHP pode ser:
    # A) {'dim_process': N, 'train': [...], 'dev': [...], 'test': [...]}  → multikey
    # B) {'dim_process': N, split: [...]}                                 → já separado
    # C) lista direta [seq1, seq2, ...]                                   → bare list
    if isinstance(raw, dict):
        dim_process = raw.get('dim_process', None)
        if split in raw:
            seqs = raw[split]
        elif 'train' in raw:         # multikey — pega o split correto
            seqs = raw.get(split, raw.get('train'))
        else:
            seqs = list(raw.values())[-1]   # fallback
    else:
        seqs = raw
        dim_process = None

    seqs = normalize_sequences(seqs)

    # Infere dim_process se não estava no pkl
    if dim_process is None:
        all_types = {e['type_event'] for seq in seqs for e in seq}
        dim_process = len(all_types)

    os.makedirs(os.path.dirname(dst_pkl) or '.', exist_ok=True)
    with open(dst_pkl, 'wb') as f:
        pickle.dump({'dim_process': dim_process, split: seqs}, f)

    return dim_process


def prepare_nhp_dataset(nhp_subdir: str, nhp_fold, dst_dir: str) -> int:
    """
    Prepara um dataset NHP para o EasyTPP:
    - nhp_subdir: pasta dentro de NHP_RAW_DIR (ex: 'data_bookorder')
    - nhp_fold:   subfold a usar (ex: 'fold1') ou None se não houver folds
    - dst_dir:    onde salvar train.pkl / dev.pkl / test.pkl

    Retorna dim_process.
    """
    os.makedirs(dst_dir, exist_ok=True)

    if nhp_fold:
        src_dir = os.path.join(NHP_RAW_DIR, nhp_subdir, nhp_fold)
    else:
        src_dir = os.path.join(NHP_RAW_DIR, nhp_subdir)

    # Detecta estrutura: arquivos diretos ou subpastas de fold
    entries = os.listdir(src_dir)
    has_pkl = any(e.endswith('.pkl') for e in entries)
    if not has_pkl:
        # Deve haver subpastas; tenta fold1 como padrão
        fold_dir = os.path.join(src_dir, 'fold1')
        if os.path.isdir(fold_dir):
            src_dir = fold_dir
            print(f"  Usando subpasta fold1 em {src_dir}")
        else:
            raise FileNotFoundError(
                f"Nenhum .pkl encontrado em {src_dir}\n"
                f"Conteúdo: {entries}"
            )

    split_files = {
        'train': 'train.pkl',
        'dev':   'dev.pkl',
        'test':  'test.pkl',
    }
    # Mapeamento alternativo para casos em que o arquivo se chama 'valid.pkl'
    for split, fname in split_files.items():
        src_pkl = os.path.join(src_dir, fname)
        if not os.path.exists(src_pkl):
            alt = 'valid.pkl' if split == 'dev' else None
            if alt and os.path.exists(os.path.join(src_dir, alt)):
                src_pkl = os.path.join(src_dir, alt)
            else:
                print(f"  AVISO: {src_pkl} não encontrado — pulando split {split}.")
                continue

        dst_pkl = os.path.join(dst_dir, f'{split}.pkl')
        dim_process = nhp_pkl_to_easytpp(src_pkl, split, dst_pkl)
        print(f"  [{split}] {src_pkl} → {dst_pkl}  (dim_process={dim_process})")

    return dim_process




def read_pkl_num_types(pkl_dir: str) -> int:
    for fname in ['train.pkl', 'dev.pkl', 'test.pkl']:
        p = os.path.join(pkl_dir, fname)
        if os.path.exists(p):
            with open(p, 'rb') as f:
                return pickle.load(f)['dim_process']
    raise FileNotFoundError(f"Nenhum pkl em {pkl_dir}")


def make_eval_pkl(src_pkl: str, src_split: str, l_test: int, dst_pkl: str) -> int:
    """Filtra sequências com len >= l_test e trunca para l_test eventos."""
    with open(src_pkl, 'rb') as f:
        data = pickle.load(f)
    dim_process, seqs = data['dim_process'], data[src_split]

    filtered = []
    for seq in seqs:
        if len(seq) >= l_test:
            trunc = [dict(e) for e in seq[:l_test]]
            for i in range(1, len(trunc)):
                trunc[i]['time_since_last_event'] = (
                    trunc[i]['time_since_start'] - trunc[i-1]['time_since_start'])
            filtered.append(trunc)

    os.makedirs(os.path.dirname(dst_pkl) or '.', exist_ok=True)
    with open(dst_pkl, 'wb') as f:
        pickle.dump({'dim_process': dim_process, src_split: filtered}, f)
    return len(filtered)


# ──────────────────────────────────────────────────────────────────
# Converte todos os datasets do NeuralHawkes Drive para EasyTPP pkl
# ──────────────────────────────────────────────────────────────────
dataset_pkl_dirs  = {}
dataset_num_types = {}

for ds_name, ds_cfg in DATASETS.items():
    print(f"\n[{ds_name}] {ds_cfg['note']}")
    local_dir = os.path.join(DATA_DIR, ds_name)

    if ds_cfg.get('pkl_dir'):                          # override manual
        if os.path.exists(ds_cfg['pkl_dir']):
            n = read_pkl_num_types(ds_cfg['pkl_dir'])
            dataset_num_types[ds_name] = n
            dataset_pkl_dirs[ds_name]  = ds_cfg['pkl_dir']
            print(f"  Override manual: {ds_cfg['pkl_dir']}  (dim={n})")
        else:
            print(f"  AVISO: pkl_dir não encontrado — pulando.")
        continue

    # Todos os datasets vêm do NeuralHawkes Drive
    if not os.path.exists(os.path.join(local_dir, 'train.pkl')):
        print(f"  Convertendo {ds_cfg['nhp_subdir']} → EasyTPP pkl...")
        n = prepare_nhp_dataset(
            ds_cfg['nhp_subdir'], ds_cfg.get('nhp_fold'), local_dir)
    else:
        print(f"  pkl já convertido, pulando.")
        n = read_pkl_num_types(local_dir)

    if ds_cfg.get('num_types'):
        n = ds_cfg['num_types']
    dataset_num_types[ds_name] = n
    dataset_pkl_dirs[ds_name]  = local_dir
    print(f"  dim_process={n}")

print(f"\nDatasets prontos  : {list(dataset_pkl_dirs.keys())}")
print(f"Num. tipos        : {dataset_num_types}")


## Célula 4 — Funções Auxiliares: Config, Runner, NLL

Funções para construir a configuração YAML do EasyTPP, instanciar o Runner e extrair o NLL.

**Notas de implementação:**
- `metrics: ['loglike', 'rmse', 'acc']` — inclui `rmse` para satisfazer o retorno de
  `base_runner.evaluate()` (que faz `return metric['rmse']`), embora usemos `_evaluate_model` diretamente.
- O YAML temporário é escrito em `/tmp/` para evitar poluir o diretório de trabalho.
- Em modo `eval`, o Runner ainda precisa de `train.pkl` para calcular estatísticas
  de `log(dt)` (media e desvio). O `train.pkl` original é copiado para o `eval_dir`.

In [ ]:
import yaml
import tempfile
import torch
import os
import pickle
import shutil

# Importacoes do EasyTPP
from easy_tpp.config_factory import Config
from easy_tpp.runner import Runner
from easy_tpp.utils import RunnerPhase


def build_runner_config_dict(
    ds_name: str,
    pkl_dir: str,
    num_event_types: int,
    model_id: str,
    stage: str,
    max_len: int,
    experiment_id: str,
    pretrained_model_dir: str = None,
) -> dict:
    """
    Constroi o dicionario de configuracao YAML para o EasyTPP Runner.

    stage: 'train' | 'eval'

    A key 'metrics' inclui 'rmse' para evitar KeyError no base_runner.evaluate(),
    que faz `return metric['rmse']`. Porem usamos _evaluate_model() diretamente
    para obter o dict completo com 'loglike'.
    """
    trainer_metrics = ['loglike', 'rmse', 'acc']

    base_cfg = {
        'stage':      stage,
        'backend':    'torch',
        'dataset_id': ds_name,
        'runner_id':  'std_tpp',
        'model_id':   model_id,
        'base_dir':   BASE_DIR,
    }
    if stage == 'eval' and pretrained_model_dir:
        base_cfg['pretrained_model_dir'] = pretrained_model_dir

    return {
        'pipeline_config_id': 'runner_config',
        'data': {
            ds_name: {
                'data_format': 'pkl',
                'train_dir':   os.path.join(pkl_dir, 'train.pkl'),
                'valid_dir':   os.path.join(pkl_dir, 'dev.pkl'),
                'test_dir':    os.path.join(pkl_dir, 'test.pkl'),
                'data_specs': {
                    'num_event_types': num_event_types,
                    'pad_token_id':    num_event_types,   # pad index = num_types
                    'padding_side':    'right',
                    'truncation_side': 'right',
                    'max_len':         max_len,
                },
            }
        },
        experiment_id: {
            'base_config': base_cfg,
            'model_config': {
                'hidden_size':   HIDDEN_SIZE,
                'time_emb_size': TIME_EMB_SIZE,
                'num_layers':    NUM_LAYERS,
                'num_heads':     NUM_HEADS,
                'dropout':       DROPOUT,
                'use_ln':        False,
                'loss_integral_num_sample_per_step': 20,
                'thinning_params': {
                    'num_seq':              10,
                    'num_sample':           1,
                    'num_exp':              500,
                    'look_ahead_time':      10,
                    'patience_counter':     5,
                    'over_sample_rate':     5,
                    'num_samples_boundary': 5,
                    'dtime_max':            5,
                },
            },
            'trainer_config': {
                'seed':          2019,
                'gpu':           GPU,
                'batch_size':    BATCH_SIZE,
                'max_epoch':     MAX_EPOCHS,
                'optimizer':     'adam',
                'learning_rate': LEARNING_RATE,
                'valid_freq':    1,
                'use_tfb':       False,
                'metrics':       trainer_metrics,
            },
        },
    }


def build_runner(cfg_dict: dict, experiment_id: str) -> Runner:
    """
    Escreve cfg_dict em um YAML temporario em /tmp/ e instancia o Runner.
    O arquivo temporario e removido apos a instanciacao.
    """
    with tempfile.NamedTemporaryFile(
        mode='w', suffix='.yaml', delete=False, dir='/tmp'
    ) as f:
        yaml.dump(cfg_dict, f, default_flow_style=False, allow_unicode=True)
        cfg_path = f.name
    try:
        runner_cfg = Config.build_from_yaml_file(
            cfg_path, experiment_id=experiment_id
        )
        return Runner.build_from_config(runner_cfg)
    finally:
        os.unlink(cfg_path)


def eval_nll(runner: Runner) -> float:
    """
    Avalia o modelo no test_loader e retorna o NLL medio por evento.

    O EasyTPP reporta 'loglike' (log-verossimilhanca, maior = melhor).
    Devolvemos NLL = -loglike (menor = melhor) para a analise comparativa.

    Usamos runner._evaluate_model(test_loader) diretamente para obter
    o dict completo de metricas, em vez de runner.evaluate() que faz
    `return metric['rmse']` e descarta o loglike.
    """
    test_loader = runner._data_loader.test_loader()
    metrics = runner._evaluate_model(test_loader)
    loglike = metrics.get('loglike', float('nan'))
    return -loglike   # NLL = -loglike


def get_checkpoint_dir(experiment_id: str) -> str:
    """Caminho padrao onde o EasyTPP salva o checkpoint apos o treino."""
    return os.path.join(BASE_DIR, experiment_id, 'models', 'saved_model')


print("Funcoes auxiliares definidas com sucesso.")

## Célula 5 — Treinamento

Treina cada combinacao `(dataset, modelo)` com `max_len = L_train`.
Se o checkpoint ja existir no disco, o treino e pulado automaticamente.

Os checkpoints ficam em `./checkpoints/<experiment_id>/models/saved_model`.

In [ ]:
import os
import pickle
import traceback

trained = {}   # (ds_name, model_id) -> caminho do checkpoint

for ds_name, pkl_dir in dataset_pkl_dirs.items():
    l_train = DATASETS[ds_name]['l_train'] or L_TRAIN
    n_types = dataset_num_types[ds_name]

    for model_id in MODELS:
        exp_id   = f'{model_id}_{ds_name}_Ltrain{l_train}'
        ckpt_dir = get_checkpoint_dir(exp_id)

        print(f"\n{'='*60}")
        print(f"TREINO  {model_id} | {ds_name} | L_train={l_train}")
        print(f"Experiment ID: {exp_id}")
        print(f"{'='*60}")

        if os.path.exists(ckpt_dir):
            print(f"  Checkpoint ja existe: {ckpt_dir}")
            print(f"  Pulando treino.")
            trained[(ds_name, model_id)] = ckpt_dir
            continue

        cfg = build_runner_config_dict(
            ds_name=ds_name,
            pkl_dir=pkl_dir,
            num_event_types=n_types,
            model_id=model_id,
            stage='train',
            max_len=l_train,
            experiment_id=exp_id,
        )
        try:
            runner = build_runner(cfg, exp_id)
            runner.train()
            trained[(ds_name, model_id)] = ckpt_dir
            print(f"  Checkpoint salvo em: {ckpt_dir}")
        except Exception as e:
            print(f"  ERRO ao treinar {model_id} em {ds_name}: {e}")
            traceback.print_exc()

print(f"\nModelos treinados com sucesso: {list(trained.keys())}")

## Célula 6 — Avaliação com Extrapolação de Comprimento

Para cada combinacao `(dataset, modelo, ratio)`:

1. Calcula `L_test = ratio * L_train`
2. Gera um pkl de teste filtrado: mantém apenas sequencias com `len >= L_test`
   e trunca cada uma para exatamente `L_test` eventos
3. Carrega o modelo treinado com `max_len=L_train`
4. Avalia com `max_len=L_test` (extrapolacao se `ratio > 1`)
5. Registra o NLL medio

**Nota:** O Runner precisa de `train.pkl` mesmo em modo eval, pois calcula
estatisticas de `log(dt)` a partir do treino para normalizar o modelo.
Copiamos o `train.pkl` original para o diretorio de avaliacao.

In [ ]:
import os
import pickle
import shutil
import traceback

results = {}   # (ds_name, model_id, ratio) -> dict com 'nll', 'l_test', 'n_seqs'

for ds_name, pkl_dir in dataset_pkl_dirs.items():
    l_train = DATASETS[ds_name]['l_train'] or L_TRAIN
    n_types = dataset_num_types[ds_name]

    for ratio in EXTRAPOLATION_RATIOS:
        l_test = ratio * l_train

        # ── Gera pkl de teste filtrado e truncado para L_test ────────
        eval_dir = os.path.join(DATA_DIR, f'{ds_name}_eval_L{l_test}')
        os.makedirs(eval_dir, exist_ok=True)

        dst_test  = os.path.join(eval_dir, 'test.pkl')
        dst_dev   = os.path.join(eval_dir, 'dev.pkl')
        dst_train = os.path.join(eval_dir, 'train.pkl')

        # test.pkl filtrado
        if not os.path.exists(dst_test):
            n_test = make_eval_pkl(
                os.path.join(pkl_dir, 'test.pkl'), 'test', l_test, dst_test
            )
            print(f"  {ds_name} L_test={l_test}: geradas {n_test} seqs de teste")
        else:
            with open(dst_test, 'rb') as f:
                n_test = len(pickle.load(f)['test'])

        # dev.pkl filtrado (necessario para o Runner)
        if not os.path.exists(dst_dev):
            make_eval_pkl(
                os.path.join(pkl_dir, 'dev.pkl'), 'dev', l_test, dst_dev
            )

        # train.pkl copiado do original (para calcular stats de dt)
        if not os.path.exists(dst_train):
            shutil.copy(os.path.join(pkl_dir, 'train.pkl'), dst_train)

        # Verifica numero de sequencias suficientes
        if n_test == 0:
            print(f"  AVISO: {ds_name} L_test={l_test} -> 0 sequencias. Ratio {ratio}x pulado.")
            continue
        if n_test < 30:
            print(f"  AVISO: {ds_name} L_test={l_test} -> apenas {n_test} seqs (baixa confianca estatistica).")

        # ── Avalia cada modelo com max_len = L_test ─────────────────
        for model_id in MODELS:
            if (ds_name, model_id) not in trained:
                print(f"  AVISO: ({ds_name}, {model_id}) sem checkpoint — pulando.")
                continue

            ckpt_dir = trained[(ds_name, model_id)]
            exp_id   = f'{model_id}_{ds_name}_eval_L{l_test}'

            tag = f"{ratio}x (L_test={l_test})"
            print(f"\n  EVAL  {model_id} | {ds_name} | {tag} | {n_test} seqs")

            cfg = build_runner_config_dict(
                ds_name=ds_name,
                pkl_dir=eval_dir,
                num_event_types=n_types,
                model_id=model_id,
                stage='eval',
                max_len=l_test,
                experiment_id=exp_id,
                pretrained_model_dir=ckpt_dir,
            )
            try:
                runner = build_runner(cfg, exp_id)
                nll    = eval_nll(runner)
                results[(ds_name, model_id, ratio)] = {
                    'nll':    nll,
                    'l_test': l_test,
                    'n_seqs': n_test,
                }
                print(f"    NLL = {nll:.4f}  (loglike = {-nll:.4f})")
            except Exception as e:
                print(f"    ERRO: {e}")
                traceback.print_exc()
                results[(ds_name, model_id, ratio)] = {
                    'nll': float('nan'), 'l_test': l_test, 'n_seqs': n_test
                }

print("\nAvaliacao completa.")
print(f"Total de (ds, model, ratio) avaliados: {len(results)}")

## Célula 7 — Resultados: Tabela e Gráficos

**Gráfico 1:** NLL absoluto por L_test, uma subplot por dataset.
Linha pontilhada cinza em `L_train` indica o limiar de extrapolacao.

**Gráfico 2:** ΔNLL em relacao ao THP como baseline (`ΔNLL = NLL_modelo - NLL_THP`).
Valores negativos indicam que o modelo supera o THP naquele comprimento.

Arquivos salvos em `./results/`:
- `extrapolation_results.csv` — tabela pivot NLL
- `extrapolation_nll.pdf` — Gráfico 1
- `extrapolation_delta_nll.pdf` — Gráfico 2

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='whitegrid', font_scale=1.1)

MODEL_COLORS  = {'THP': '#e05c5c',  'RoTHP': '#5c9ee0',  'HoTHP': '#5ccc88'}
MODEL_STYLES  = {'THP': '--',       'RoTHP': '-.',        'HoTHP': '-'}
MODEL_MARKERS = {'THP': 'o',        'RoTHP': 's',         'HoTHP': '^'}


def results_to_df(results: dict) -> pd.DataFrame:
    rows = []
    for (ds, model, ratio), v in results.items():
        rows.append({
            'Dataset': ds,
            'Model':   model,
            'Ratio':   f'{ratio}x',
            'L_test':  v['l_test'],
            'NLL':     round(v['nll'], 4),
            'N_seqs':  v['n_seqs'],
        })
    return pd.DataFrame(rows).sort_values(['Dataset', 'Model', 'L_test'])


if not results:
    print("Nenhum resultado disponivel. Execute as celulas 5 e 6 primeiro.")
else:
    df = results_to_df(results)

    # ── Tabela pivot ─────────────────────────────────────────────────
    print("\n=== NLL por modelo, dataset e ratio (menor e melhor) ===\n")
    pivot = df.pivot_table(values='NLL', index=['Dataset', 'Model'], columns='Ratio')
    # Reordena colunas por L_test crescente
    ratio_order = [f'{r}x' for r in EXTRAPOLATION_RATIOS]
    pivot = pivot[[c for c in ratio_order if c in pivot.columns]]
    print(pivot.to_string())

    csv_path = os.path.join(RESULTS_DIR, 'extrapolation_results.csv')
    pivot.to_csv(csv_path)
    print(f"\nTabela salva em {csv_path}")

    active_datasets = df['Dataset'].unique()
    n_ds = len(active_datasets)

    # ── Grafico 1: NLL absoluto por L_test ──────────────────────────
    fig, axes = plt.subplots(1, n_ds, figsize=(6 * n_ds, 5), squeeze=False)

    for col, ds_name in enumerate(active_datasets):
        ax      = axes[0][col]
        l_train = DATASETS[ds_name]['l_train'] or L_TRAIN
        ds_df   = df[df['Dataset'] == ds_name]

        for model_id in MODELS:
            m_df = ds_df[ds_df['Model'] == model_id].sort_values('L_test')
            if m_df.empty:
                continue
            ax.plot(
                m_df['L_test'], m_df['NLL'],
                label=model_id,
                color=MODEL_COLORS[model_id],
                linestyle=MODEL_STYLES[model_id],
                marker=MODEL_MARKERS[model_id],
                linewidth=2, markersize=7,
            )

        ax.axvline(x=l_train, color='gray', linestyle=':', linewidth=1.5,
                   label=f'L_train={l_train}')
        ax.set_title(ds_name.capitalize(), fontsize=13)
        ax.set_xlabel('L_test (events)', fontsize=11)
        ax.set_ylabel('NLL (lower is better)', fontsize=11)
        ax.set_xticks([r * l_train for r in EXTRAPOLATION_RATIOS])
        ax.set_xticklabels([f'{r}x\n({r*l_train})' for r in EXTRAPOLATION_RATIOS])
        ax.legend(fontsize=10)

    plt.suptitle('THP vs RoTHP vs HoTHP — Length Extrapolation', fontsize=14, y=1.02)
    plt.tight_layout()
    out1 = os.path.join(RESULTS_DIR, 'extrapolation_nll.pdf')
    plt.savefig(out1, bbox_inches='tight', dpi=150)
    print(f"Figura 1 (NLL absoluto) salva em {out1}")
    plt.show()

    # ── Grafico 2: Delta NLL vs THP como baseline ─────────────────
    fig2, axes2 = plt.subplots(1, n_ds, figsize=(6 * n_ds, 5), squeeze=False)

    for col, ds_name in enumerate(active_datasets):
        ax      = axes2[0][col]
        l_train = DATASETS[ds_name]['l_train'] or L_TRAIN
        ds_df   = df[df['Dataset'] == ds_name]
        thp_nll = ds_df[ds_df['Model'] == 'THP'].set_index('L_test')['NLL']

        if thp_nll.empty:
            ax.set_title(f'{ds_name} — THP sem resultados')
            continue

        for model_id in ['RoTHP', 'HoTHP']:
            m_df = ds_df[ds_df['Model'] == model_id].sort_values('L_test')
            if m_df.empty:
                continue
            delta = [
                row['NLL'] - thp_nll.get(row['L_test'], float('nan'))
                for _, row in m_df.iterrows()
            ]
            ax.plot(
                m_df['L_test'], delta,
                label=f'{model_id} - THP',
                color=MODEL_COLORS[model_id],
                linestyle=MODEL_STYLES[model_id],
                marker=MODEL_MARKERS[model_id],
                linewidth=2, markersize=7,
            )

        ax.axhline(y=0, color=MODEL_COLORS['THP'], linestyle='--',
                   linewidth=1.5, label='THP (baseline = 0)')
        ax.axvline(x=l_train, color='gray', linestyle=':', linewidth=1.5)

        # Regiao positiva = RoTHP/HoTHP pior que THP
        l_test_vals = [r * l_train for r in EXTRAPOLATION_RATIOS]
        ax.fill_between(
            [min(l_test_vals), max(l_test_vals)], [-0.5, -0.5], [0, 0],
            alpha=0.05, color='green',
        )
        ax.set_title(f'{ds_name.capitalize()} (delta NLL vs THP)', fontsize=13)
        ax.set_xlabel('L_test (events)', fontsize=11)
        ax.set_ylabel('delta NLL vs THP (lower is better)', fontsize=11)
        ax.set_xticks(l_test_vals)
        ax.set_xticklabels([f'{r}x\n({r*l_train})' for r in EXTRAPOLATION_RATIOS])
        ax.legend(fontsize=10)

    plt.suptitle('RoTHP and HoTHP advantage over THP in length extrapolation',
                 fontsize=14, y=1.02)
    plt.tight_layout()
    out2 = os.path.join(RESULTS_DIR, 'extrapolation_delta_nll.pdf')
    plt.savefig(out2, bbox_inches='tight', dpi=150)
    print(f"Figura 2 (delta NLL) salva em {out2}")
    plt.show()

    print("\nResultados completos:")
    print(df.to_string(index=False))

## Fontes dos Datasets

**Drive público** (Hugo Ramos Soares, 2026):  
`https://drive.google.com/drive/folders/1bxwawkeRmsPpDmJH0oMeYREno5NZttxu`

Dados originados de: Mei & Eisner, *The Neural Hawkes Process*, NeurIPS 2017.

| Pasta | Dataset | Num. tipos |
|-------|---------|-----------|
| `so` | StackOverflow | 22 |
| `retweet` | Retweet | 3 |
| `meme` | Memetrack | — |
| `mimic` | Mimic-II | — |

---

### L_train por dataset

| Dataset | L_train | 1× | 2× | 5× | 10× |
|---------|---------|-----|-----|-----|------|
| StackOverflow | 10 | 10 | 20 | 50 | 100 |
| Retweet | 10 | 10 | 20 | 50 | 100 |
| Memetrack | 50 | 50 | 100 | 250 | 500 |
| Mimic-II | 10 | 10 | 20 | 50 | 100 |